In [5]:
%pip install pandas scikit-learn


Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 159.3 kB/s  0:01:00a 0:00:02m eta 0:00:04
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pandas]━━━━ 5/6 [pandas]scikit-learn]
Note: you may need to res

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

def preprocess_data(data):
    passenger_ids = data['PassengerId'] if 'PassengerId' in data.columns else None
    data = data.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], errors='ignore')
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    data['Embarked'] = data['Embarked'].fillna('S')
    data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True)
    return data, passenger_ids

train_cleaned, _ = preprocess_data(train_df)
test_cleaned, test_ids = preprocess_data(test_df)

X = train_cleaned.drop(columns=['Survived'])
y = train_cleaned['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

val_predictions = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)

print(f"Realistic Local Validation Accuracy: {val_accuracy:.2%}\n")
print("Detailed Performance Report:")
print(classification_report(y_val, val_predictions))

test_cleaned = test_cleaned.reindex(columns=X.columns, fill_value=0)
final_preds = rf_model.predict(test_cleaned)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': final_preds
})
submission.to_csv('submission.csv', index=False)
print("\nSuccess! Created 'submission.csv' in your workspace folder.")


Realistic Local Validation Accuracy: 81.56%

Detailed Performance Report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.84      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179


Success! Created 'submission.csv' in your workspace folder.
